In [1]:
# Preeliminary: Load val dataset

import sys
from pathlib import Path

# Notebooks live in notebooks/; project packages are at the repo root.
sys.path.insert(0, str(Path.cwd().resolve().parent))

import os
from torch.utils.data import DataLoader
from data.imagenet import ImageNetDataset
from data.transforms.transforms import build_train_batch_transforms, build_val_transforms
from engine.validator import Validator
from metrics.metricHistory import DictHistoryMetrics
from metrics.metricLoss import MetricLoss
from metrics.top1acc import Top1AccMetric
from models.backbones.convnext import ConvNextV1
from timm.loss import SoftTargetCrossEntropy
import torch
import torch.nn as nn

from models.backbones.delta_convnext import DeltaConvNext

BATCH_SIZE = 128
num_workers = 10
val_dataset = ImageNetDataset(split="validation", transforms=build_val_transforms())
def validateModel(blockList: tuple[int, ...]):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DeltaConvNext()
    stateDict = torch.load("../outputs/outputs/convnextv1_imagenet/weights/last.pth", map_location=device)["model"]
    model.load_state_dict(stateDict, strict=False)
    model.rewire(blockList)
    model.eval()

    # train loader and val loader of imagenet with huggingface datasets


    print(f"Dataloader workers: {num_workers}")

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True,
    )

    validator = Validator(
        model=model,
        criterion=SoftTargetCrossEntropy(),
        device=device,
        val_loader=val_loader,
        batch_transforms=build_train_batch_transforms(),
        num_classes=1000,
        amp=False,
    )
    val_histoy_metrics = DictHistoryMetrics(Path(f"outputs/convNext_validation"), split="val")
    val_histoy_metrics.addHistoryMetric("top1acc", Top1AccMetric)
    val_histoy_metrics.addHistoryMetric("loss", MetricLoss, higher_is_better=False)
    validator.validate(val_histoy_metrics)


### Preeliminar: Test that the new class imports correctly the model and gives us same accuracy and loss

In [13]:
validateModel([i for i in range(9)])

0 1 2 3 4 5 6 7 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:52<00:00,  3.49it/s, loss=0.889]

val_top1acc: 0.80962
val_loss: 0.8878384390640259


Correct!
| acc | loss |
|-----:|-----:|
| 0.80962 | 0.88784 |

In [ ]:
for i in range(9):
    print(f"Block {i}")
    validateModel([i])

Block 0
Acc: 0.00074    loss: 7.554850865325927

Block 1
Acc: 0.00112    loss: 7.573008715057373

Block 2
Acc: 0.00084    loss: 7.577597916564941

Block 3
Acc: 0.00114    loss: 7.5815993434143065

Block 4
Acc: 0.00134    loss: 7.4202857620239255

Block 5
Acc: 0.00052    loss: 7.551064516906738

Block 6
Acc: 0.0006     loss: 7.5387580838012695

Block 7
Acc: 0.00096    loss: 7.592989510650635

Block 8
Acc: 0.00114    loss: 7.521969194488525

## First abalation study: Ommision

We'll ommit one block and see how well it works. After that, we will keep ommited the one that gived us most accuracy

In [ ]:
for i in range(9):
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i])

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 1 | 0.80484 | 0.91299 |
| 2 | 0.80504 | 0.90643 |
| 3 | 0.80390 | 0.90777 |
| 4 | 0.80556 | 0.90208 |
| 5 | 0.80450 | 0.90326 |
| 6 | 0.80366 | 0.90468 |
| 7 | 0.80120 | 0.91863 |
| 8 | 0.79708 | 0.93022 |

When removing the block nº 0, the accuracy lowers the least.

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |

In [ ]:
OMMIT_ALWAYS = [0]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

| block | acc | loss |
|------:|-----:|-----:|
| 1 | 0.79960 | 0.93836 |
| 2 | 0.79548 | 0.95398 |
| 3 | 0.80070 | 0.93519 |
| 4 | 0.80116 | 0.92604 |
| 5 | 0.79964 | 0.92891 |
| 6 | 0.80074 | 0.92737 |
| 7 | 0.79790 | 0.94092 |
| 8 | 0.79434 | 0.95358 |

Best acc: Ommiting block 0 + block 4:
| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |

In [ ]:
OMMIT_ALWAYS = [0,4]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

| block | acc | loss |
|------:|-----:|-----:|
| 1 | 0.79522 | 0.96175 |
| 2 | 0.78854 | 0.98491 |
| 3 | 0.78978 | 0.98180 |
| 5 | 0.79290 | 0.95796 |
| 6 | 0.79478 | 0.94952 |
| 7 | 0.79168 | 0.96085 |
| 8 | 0.78816 | 0.97861 |

Best acc: Ommiting block 0 + block 4 + block 1
| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |

In [2]:
OMMIT_ALWAYS = [0,4,1]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 2
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:27<00:00,  4.47it/s, loss=1.08] 


val_top1acc: 0.77284
val_loss: 1.0580339723587036
Ommiting block 3
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:31<00:00,  4.28it/s, loss=1.05] 


val_top1acc: 0.77544
val_loss: 1.0502075818252563
Ommiting block 5
Model rewired from old ConvNext!
Dataloader workers: 10


...

val_top1acc: 0.78572
val_loss: 0.9973198097610474
Ommiting block 7
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:44<00:00,  3.73it/s, loss=0.993]


val_top1acc: 0.78356
val_loss: 1.001802107887268
Ommiting block 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:50<00:00,  3.55it/s, loss=0.996]

val_top1acc: 0.77868
val_loss: 1.0270762660980224


| block | acc | loss |
|------:|-----:|-----:|
| 2 | 0.77284 | 1.05803 |
| 3 | 0.77544 | 1.05021 |
| 5 | 0.78320 | 1.00435 |
| 6 | 0.78572 | 0.99732 |
| 7 | 0.78356 | 1.00180 |
| 8 | 0.77868 | 1.02708 |

Best acc: Ommiting block 0 + block 4 + block 1 + block 6
| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |

In [3]:
OMMIT_ALWAYS = [0,4,1,6]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 2
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:22<00:00,  4.72it/s, loss=1.11] 


val_top1acc: 0.76058
val_loss: 1.111261757850647
Ommiting block 3
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:33<00:00,  4.20it/s, loss=1.06] 


val_top1acc: 0.76348
val_loss: 1.1078787009811402
Ommiting block 5
Model rewired from old ConvNext!
Dataloader workers: 10


...

val_top1acc: 0.76656
val_loss: 1.0879933555603027
Ommiting block 7
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:41<00:00,  3.85it/s, loss=1.01] 


val_top1acc: 0.76668
val_loss: 1.0723376342773439
Ommiting block 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:42<00:00,  3.80it/s, loss=1.02] 

val_top1acc: 0.76446
val_loss: 1.0830326739501952


| block | acc | loss |
|------:|-----:|-----:|
| 2 | 0.76058 | 1.11126 |
| 3 | 0.76348 | 1.10788 |
| 5 | 0.76656 | 1.08799 |
| 7 | 0.76668 | 1.07234 |
| 8 | 0.76446 | 1.08303 |

**Best acc:** Ommiting block 0 + block 4 + block 1 + block 6 + block 7

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |

In [7]:
OMMIT_ALWAYS = [0,4,1,6,7]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 2
3 5 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:16<00:00,  5.13it/s, loss=1.19]


val_top1acc: 0.73316
val_loss: 1.226369291191101
Ommiting block 3
2 5 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:22<00:00,  4.72it/s, loss=1.18] 


val_top1acc: 0.73338
val_loss: 1.2325875959777832
Ommiting block 5
2 3 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:24<00:00,  4.65it/s, loss=1.17]


val_top1acc: 0.72366
val_loss: 1.2670782907104492
Ommiting block 8
2 3 5
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:25<00:00,  4.60it/s, loss=1.32]

val_top1acc: 0.68766
val_loss: 1.4604508153915405


| block | acc | loss |
|------:|-----:|-----:|
| 2 | 0.73316 | 1.22637 |
| 3 | 0.73338 | 1.23259 |
| 5 | 0.72366 | 1.26708 |
| 8 | 0.68766 | 1.46045 |

**Best acc:** Ommiting block 0 + block 4 + block 1 + block 6 + block 7 + block **3**

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |
| 0 + 4 + 1 + 6 + 7 + 3 | 0.73338 | 1.23259 |

In [8]:
OMMIT_ALWAYS = [0,4,1,6,7,3]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 2
5 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:16<00:00,  5.12it/s, loss=1.57]


val_top1acc: 0.62486
val_loss: 1.7578512495040894
Ommiting block 5
2 8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:20<00:00,  4.84it/s, loss=1.66]


val_top1acc: 0.6193
val_loss: 1.786969148826599
Ommiting block 8
2 5
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:23<00:00,  4.70it/s, loss=1.75]

val_top1acc: 0.61092
val_loss: 1.860664619102478


| block | acc | loss |
|------:|-----:|-----:|
| 2 | 0.62486 | 1.75785 |
| 5 | 0.61930 | 1.78697 |
| 8 | 0.61092 | 1.86066 |

**Best acc:** Ommiting block 0 + block 4 + block 1 + block 6 + block 7 + block 3 + block **2**

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |
| 0 + 4 + 1 + 6 + 7 + 3 | 0.73338 | 1.23259 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 | 0.62486 | 1.75785 |

In [9]:
OMMIT_ALWAYS = [0,4,1,6,7,3,2]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 5
8
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:09<00:00,  5.59it/s, loss=3.16]


val_top1acc: 0.36154
val_loss: 3.270049895095825
Ommiting block 8
5
Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:16<00:00,  5.08it/s, loss=2.65]

val_top1acc: 0.43118
val_loss: 2.8857251356506346


| block | acc | loss |
|------:|-----:|-----:|
| 5 | 0.36154 | 3.27005 |
| 8 | 0.43118 | 2.88573 |

**Best acc:** Ommiting block 0 + block 4 + block 1 + block 6 + block 7 + block 3 + block 2 + block **8**

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |
| 0 + 4 + 1 + 6 + 7 + 3 | 0.73338 | 1.23259 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 | 0.62486 | 1.75785 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 + 8 | 0.43118 | 2.88573 |

In [10]:
OMMIT_ALWAYS = [0,4,1,6,7,3,2,8]
for i in range(9):
    if i in OMMIT_ALWAYS:
        continue
    print(f"Ommiting block {i}")
    validateModel([j for j in range(9) if j != i and j not in OMMIT_ALWAYS])

Ommiting block 5

Model rewired from old ConvNext!
Dataloader workers: 10


100%|██████████| 391/391 [01:08<00:00,  5.74it/s, loss=5.55]

val_top1acc: 0.06622
val_loss: 5.6655239674377444


| block | acc | loss |
|------:|-----:|-----:|
| 5 | 0.06622 | 5.66552 |

**Best acc:** Ommiting block 0 + block 4 + block 1 + block 6 + block 7 + block 3 + block 2 + block 8 + block **5**

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |
| 0 + 4 + 1 + 6 + 7 + 3 | 0.73338 | 1.23259 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 | 0.62486 | 1.75785 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 + 8 | 0.43118 | 2.88573 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 + 8 + 5 | 0.06622 | 5.66552 |

### Final result of the 1st abalation study regarding ommiting blocks

| block | acc | loss |
|------:|-----:|-----:|
| None | 0.80962 | 0.88784 |
| 0 | 0.80666 | 0.90842 |
| 0 + 4 | 0.80116 | 0.92604 |
| 0 + 4 + 1 | 0.79522 | 0.96175 |
| 0 + 4 + 1 + 6 | 0.78572 | 0.96175 |
| 0 + 4 + 1 + 6 + 7 | 0.76668 | 1.07234 |
| 0 + 4 + 1 + 6 + 7 + 3 | 0.73338 | 1.23259 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 | 0.62486 | 1.75785 |
| 0 + 4 + 1 + 6 + 7 + 3 + 2 + 8 | 0.43118 | 2.88573 |
| All | 0.06622 | 5.66552 |

The error was very similar whenever we ommit any of the blocks on the first stages, when we went skipping more and more blocks, the differences augmented on the performance of the loss.

It is interesting that even when dropping half the stage, the network just drops ~4% of accuracy.

Of course, there should be some combinations that can perform better perofrmace with the same number of block drops. We've tried only 10 combinations of a total of the 512 posibilities just to make a fast test.

## Second abalation study: Repetition of blocks

In [ ]:
validateModel([5,5])
validateModel([5,5,5])
validateModel([5,5,5,5])
validateModel([5,5,5,5,5])
validateModel([5,5,5,5,5,5])

| config | acc | loss |
|--------|-----:|-----:|
| [5, 5] | 0.52158 | 2.31553 |
| [5, 5, 5] | 0.51928 | 2.32149 |
| [5, 5, 5, 5] | 0.47924 | 2.55349 |
| [5, 5, 5, 5, 5] | 0.42044 | 2.90247 |
| [5, 5, 5, 5, 5, 5] | 0.35516 | 3.31074 |

## Third abalation study: Order

In [12]:
validateModel([8,0])
validateModel([8,4,0])
validateModel([8,5,3,0])
validateModel([8,6,4,3,0])
validateModel([8,7,6,4,3,0])

8 0
Model rewired from old ConvNext!
Dataloader workers: 10


 75%|███████▌  | 295/391 [00:55<00:18,  5.12it/s, loss=3.05]